[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/para_casa/S11_02_transfer_learning.ipynb)

# S11 · 02 — Transfer learning y fine-tuning

Sprint 11 · UD 4.5

**Contexto TurisData.** Entrenar una CNN desde cero necesita miles de fotos etiquetadas, y el cliente tiene solo unos cientos. La solución habitual es **reutilizar una red ya entrenada** en otra tarea y adaptarla: es el *transfer learning*.

**Este cuaderno tiene 2 ejercicios y un mini-reto.** El ejercicio 1 es el mismo que el del cuaderno «en clase» (allí, el 1).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import functools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)
print("Listo ✔")

### 👀 A2 · EJEMPLO — Cargar el generador de imágenes sintéticas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué dos tareas genera el notebook y para qué sirve cada una?
2. ¿Qué tamaño tienen las imágenes?
3. ¿Son fotos reales?

In [ ]:
from PIL import Image, ImageDraw

CLASES_ALOJ = ["hotel", "apartamentos", "casa_rural", "camping", "piscina"]
CLASES_FORMAS = ["rayas_horizontales", "rayas_verticales", "diagonales", "cuadricula", "puntos", "circulo", "rectangulo", "triangulo"]


def _color(rng, base, var=30):
    """Color aleatorio cercano a `base` (tupla RGB)."""
    return tuple(int(np.clip(c + rng.integers(-var, var + 1), 0, 255)) for c in base)


def _final(img, rng, ruido=8):
    """Convierte a uint8 con un poco de ruido y variación de brillo."""
    a = np.asarray(img, dtype="float32") * rng.uniform(0.85, 1.15)
    a = a + rng.normal(0, ruido, a.shape)
    return np.clip(a, 0, 255).astype("uint8")


def _ventanas(d, x0, y0, x1, y1, filas, cols, color, margen):
    """Cuadrícula de ventanas dentro del rectángulo (x0, y0, x1, y1)."""
    ancho, alto = (x1 - x0) / cols, (y1 - y0) / filas
    for f in range(filas):
        for c in range(cols):
            a, b = x0 + c * ancho + margen * ancho, y0 + f * alto + margen * alto
            d.rectangle([a, b, a + ancho * (1 - 2 * margen), b + alto * (1 - 2 * margen)], fill=color)


def dibujar_alojamiento(clase, rng, tam=48, ruido=8):
    """Dibujo esquemático (tam×tam×3, uint8) de un tipo de alojamiento. Datos SINTÉTICOS, no fotos reales."""
    E = 3                                   # supermuestreo para suavizar bordes
    S = tam * E
    img = Image.new("RGB", (S, S), _color(rng, (170, 205, 235), 25))          # cielo
    d = ImageDraw.Draw(img)
    if rng.random() < 0.5:                                                    # sol opcional
        cx, cy, r = rng.uniform(0.1, 0.9) * S, rng.uniform(0.08, 0.25) * S, rng.uniform(0.05, 0.08) * S
        d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(250, 235, 120))
    hor = rng.uniform(0.62, 0.72) * S                                         # línea del horizonte
    verde = _color(rng, (120, 170, 90), 25)
    if clase == "piscina":
        d.rectangle([0, hor * 0.6, S, S], fill=_color(rng, (225, 210, 170), 15))   # terraza
        x0, x1 = rng.uniform(0.08, 0.2) * S, rng.uniform(0.75, 0.92) * S
        y0, y1 = rng.uniform(0.5, 0.58) * S, rng.uniform(0.82, 0.94) * S
        d.rectangle([x0 - 3, y0 - 3, x1 + 3, y1 + 3], fill=(245, 245, 245))
        d.rectangle([x0, y0, x1, y1], fill=_color(rng, (60, 150, 210), 20))
        for k in range(3):
            yy = y0 + (k + 1) * (y1 - y0) / 4
            d.line([x0 + 6, yy, x1 - 6, yy], fill=(150, 210, 240), width=2)
        for k in range(2):                                                    # tumbonas
            tx = rng.uniform(0.1, 0.8) * S
            d.rectangle([tx, hor * 0.72, tx + 0.1 * S, hor * 0.72 + 0.04 * S], fill=_color(rng, (240, 120, 60), 30))
    else:
        d.rectangle([0, hor, S, S], fill=verde)                                # césped
    if clase == "hotel":
        w, h = rng.uniform(0.30, 0.42) * S, rng.uniform(0.5, 0.66) * S
        x0 = rng.uniform(0.1 * S, 0.9 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (220, 200, 170)))
        _ventanas(d, x0, y0 + 0.02 * S, x0 + w, hor, int(rng.integers(4, 6)), 3, (60, 90, 140), 0.22)
    elif clase == "apartamentos":
        w, h = rng.uniform(0.7, 0.86) * S, rng.uniform(0.28, 0.38) * S
        x0 = rng.uniform(0.02 * S, 0.98 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (235, 225, 205)))
        _ventanas(d, x0, y0, x0 + w, hor + 0.05 * S, 2, int(rng.integers(5, 7)), (70, 100, 150), 0.25)
        for k in (1, 2):                                                       # balcones (líneas horizontales)
            yy = y0 + k * (hor + 0.05 * S - y0) / 2
            d.line([x0, yy, x0 + w, yy], fill=(120, 110, 100), width=2)
    elif clase == "casa_rural":
        w, h = rng.uniform(0.32, 0.44) * S, rng.uniform(0.2, 0.28) * S
        x0 = rng.uniform(0.1 * S, 0.9 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (235, 225, 200)))
        d.polygon([(x0 - 0.04 * S, y0), (x0 + w + 0.04 * S, y0), (x0 + w / 2, y0 - rng.uniform(0.14, 0.2) * S)],
                  fill=_color(rng, (170, 70, 50), 25))
        d.rectangle([x0 + w * 0.42, y0 + h * 0.4, x0 + w * 0.58, hor + 0.05 * S], fill=(110, 70, 40))
        d.rectangle([x0 + w * 0.1, y0 + h * 0.2, x0 + w * 0.3, y0 + h * 0.5], fill=(70, 100, 150))
    elif clase == "camping":
        d.rectangle([0, hor - 0.08 * S, S, S], fill=verde)
        w, h = rng.uniform(0.34, 0.5) * S, rng.uniform(0.28, 0.4) * S
        x0 = rng.uniform(0.05 * S, 0.95 * S - w); base = hor + 0.1 * S
        c = _color(rng, [(240, 140, 40), (60, 120, 200), (230, 200, 60)][int(rng.integers(0, 3))], 20)
        d.polygon([(x0, base), (x0 + w, base), (x0 + w / 2, base - h)], fill=c)
        d.polygon([(x0 + w * 0.4, base), (x0 + w * 0.6, base), (x0 + w / 2, base - h * 0.5)], fill=(50, 40, 40))
    if clase != "piscina" and rng.random() < 0.6:                              # árbol de adorno
        tx = rng.choice([rng.uniform(0.03, 0.12), rng.uniform(0.88, 0.97)]) * S
        d.rectangle([tx - 2, hor - 0.02 * S, tx + 2, hor + 0.12 * S], fill=(100, 70, 40))
        d.ellipse([tx - 0.07 * S, hor - 0.16 * S, tx + 0.07 * S, hor + 0.01 * S], fill=(50, 120, 60))
    return _final(img.resize((tam, tam), Image.LANCZOS), rng, ruido)


def dibujar_forma(clase, rng, tam=48):
    """Formas y texturas abstractas (tarea auxiliar de preentrenamiento). Devuelve tam×tam×3 uint8."""
    fondo, tinta = _color(rng, (rng.integers(20, 235),) * 3, 20), _color(rng, tuple(int(v) for v in rng.integers(0, 256, 3)), 10)
    if np.abs(np.array(fondo, dtype=int) - np.array(tinta, dtype=int)).sum() < 150:      # asegura contraste
        tinta = tuple(255 - c for c in fondo)
    E = 2
    S = tam * E
    img = Image.new("RGB", (S, S), fondo)
    d = ImageDraw.Draw(img)
    paso = int(rng.integers(6, 12)) * E
    off = int(rng.integers(0, paso))
    if clase == "rayas_horizontales":
        for y in range(off, S, paso): d.rectangle([0, y, S, y + paso // 2], fill=tinta)
    elif clase == "rayas_verticales":
        for x in range(off, S, paso): d.rectangle([x, 0, x + paso // 2, S], fill=tinta)
    elif clase == "diagonales":
        for k in range(-S, S, paso): d.line([k + off, 0, k + off + S, S], fill=tinta, width=max(2, paso // 3))
    elif clase == "cuadricula":
        for y in range(off, S, paso): d.line([0, y, S, y], fill=tinta, width=2)
        for x in range(off, S, paso): d.line([x, 0, x, S], fill=tinta, width=2)
    elif clase == "puntos":
        r = max(2, paso // 4)
        for y in range(off, S, paso):
            for x in range(off, S, paso): d.ellipse([x - r, y - r, x + r, y + r], fill=tinta)
    else:
        r = rng.uniform(0.18, 0.38) * S
        cx, cy = rng.uniform(r, S - r), rng.uniform(r, S - r)
        if clase == "circulo": d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=tinta)
        elif clase == "rectangulo": d.rectangle([cx - r, cy - r * rng.uniform(0.5, 1), cx + r, cy + r * rng.uniform(0.5, 1)], fill=tinta)
        else: d.polygon([(cx - r, cy + r), (cx + r, cy + r), (cx, cy - r)], fill=tinta)
    return _final(img.resize((tam, tam), Image.LANCZOS), rng)


def generar_lote(funcion, clases, n_por_clase, semilla=42, tam=48):
    """Devuelve (X uint8 [n, tam, tam, 3], y int [n]) con n_por_clase imágenes por clase, mezcladas."""
    rng = np.random.default_rng(semilla)
    X = np.stack([funcion(c, rng, tam) for c in clases for _ in range(n_por_clase)])
    y = np.repeat(np.arange(len(clases)), n_por_clase)
    orden = rng.permutation(len(y))
    return X[orden], y[orden]


PARECIDAS = {"hotel": "apartamentos", "apartamentos": "hotel", "casa_rural": "camping", "camping": "casa_rural", "piscina": "camping"}


def crear_dataset_carpetas(carpeta, n_por_clase=100, semilla=7, tam=48, ruido=8, prop_ambiguas=0.0):
    """Escribe PNG en carpeta/<clase>/img_000.png (estructura que espera image_dataset_from_directory).

    prop_ambiguas: fracción de imágenes cuyo dibujo corresponde a una clase parecida (PARECIDAS) pero se guarda
    en la carpeta de la otra: simula etiquetas dudosas o erróneas. Se anotan en carpeta/_ambiguas.csv (solo docente).
    """
    import os
    rng = np.random.default_rng(semilla)
    ambiguas = []
    for clase in CLASES_ALOJ:
        os.makedirs(os.path.join(carpeta, clase), exist_ok=True)
        for i in range(n_por_clase):
            dibujo = PARECIDAS[clase] if rng.random() < prop_ambiguas else clase
            if dibujo != clase:
                ambiguas.append(f"{clase}/img_{i:03d}.png,{dibujo}")
            Image.fromarray(dibujar_alojamiento(dibujo, rng, tam, ruido)).save(os.path.join(carpeta, clase, f"img_{i:03d}.png"))
    with open(os.path.join(carpeta, "_ambiguas.csv"), "w") as f:
        f.write("archivo,dibujo_real\n" + "\n".join(ambiguas) + "\n")
    return carpeta

### 👀 A3 · EJEMPLO — Ver las dos tareas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Qué tipo de imágenes hay en la fila superior y en la inferior?
2. ¿Qué rasgos visuales comparten las dos tareas?
3. ¿Por qué se usa una tarea auxiliar sintética?

In [ ]:
X_f, y_f = generar_lote(dibujar_forma, CLASES_FORMAS, 6, semilla=1)
X_a, y_a = generar_lote(dibujar_alojamiento, CLASES_ALOJ, 6, semilla=1)
fig, ejes = plt.subplots(2, 10, figsize=(14, 3.2))
for i in range(10):
    ejes[0, i].imshow(X_f[i]); ejes[0, i].set_title(CLASES_FORMAS[y_f[i]], fontsize=7); ejes[0, i].axis("off")
    ejes[1, i].imshow(X_a[i]); ejes[1, i].set_title(CLASES_ALOJ[y_a[i]], fontsize=7); ejes[1, i].axis("off")
plt.suptitle("Arriba: tarea auxiliar (formas y texturas) · Abajo: tarea real (tipos de alojamiento esquemáticos)")
plt.tight_layout(); plt.show()

### 👀 A4 · EJEMPLO — Crear el conjunto de entrenamiento, validación y prueba

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Cuántas imágenes hay para entrenar, validar y probar?
2. ¿Por qué las imágenes llevan tanto ruido?
3. ¿Por qué cada conjunto usa una semilla distinta?

In [ ]:
N_POR_CLASE = 6
alojamiento = functools.partial(dibujar_alojamiento, ruido=40)     # imágenes "sucias": mucho ruido, como fotos de mala calidad
X_ent, y_ent = generar_lote(alojamiento, CLASES_ALOJ, N_POR_CLASE, semilla=10)
X_val, y_val = generar_lote(alojamiento, CLASES_ALOJ, 40, semilla=11)
X_pru, y_pru = generar_lote(alojamiento, CLASES_ALOJ, 60, semilla=12)
print("entrenamiento", X_ent.shape, "· validación", X_val.shape, "· prueba", X_pru.shape)

### 👀 A5 · EJEMPLO — Entrenar una CNN desde cero con 30 imágenes

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Qué exactitud logra la CNN desde cero?
2. ¿Qué es la «base» y qué es la «cabeza»?
3. ¿Por qué esta división es la clave del *transfer learning*?

In [ ]:
def construir_base(nombre="base"):
    entrada = keras.layers.Input(shape=(48, 48, 3))
    x = keras.layers.Rescaling(1 / 255)(entrada)                     # la propia red normaliza los píxeles
    for i, f in enumerate((16, 32, 64), start=1):
        x = keras.layers.Conv2D(f, 3, padding="same", activation="relu", name=f"bloque{i}_conv")(x)
        x = keras.layers.MaxPooling2D(name=f"bloque{i}_pool")(x)
    x = keras.layers.Flatten(name="aplanar")(x)                       # 6×6×64 = 2304 rasgos
    return keras.Model(entrada, x, name=nombre)

def con_cabeza(base, n_clases, dropout=0.2, tasa=1e-3):
    """Base + cabeza nueva (Dropout y Dense softmax), ya compilado."""
    entrada = keras.layers.Input(shape=(48, 48, 3))
    x = base(entrada)
    x = keras.layers.Dropout(dropout)(x)
    salida = keras.layers.Dense(n_clases, activation="softmax", name="cabeza")(x)
    modelo = keras.Model(entrada, salida)
    modelo.compile(keras.optimizers.Adam(tasa), "sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo

def entrenar(modelo, epocas=30, semilla=SEMILLA, X=None, y=None):
    keras.utils.set_random_seed(semilla)
    X = X_ent if X is None else X
    y = y_ent if y is None else y
    return modelo.fit(X, y, validation_data=(X_val, y_val), epochs=epocas, batch_size=16, verbose=0)

keras.utils.set_random_seed(SEMILLA)
desde_cero = con_cabeza(construir_base("base_desde_cero"), 5)
h_cero = entrenar(desde_cero, epocas=40)
acc_cero = desde_cero.evaluate(X_pru, y_pru, verbose=0)[1]
print(f"CNN desde cero con {len(X_ent)} imágenes → exactitud en prueba: {acc_cero:.3f}")

### 👀 A6 · EJEMPLO — Preentrenar la base con formas y texturas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Cuántos parámetros tiene la base?
2. ¿Qué exactitud logra en la tarea auxiliar?
3. ¿En la vida real, con qué se preentrenaría la base?

In [ ]:
X_aux, y_aux = generar_lote(dibujar_forma, CLASES_FORMAS, 400, semilla=3)
X_aux_v, y_aux_v = generar_lote(dibujar_forma, CLASES_FORMAS, 50, semilla=4)

keras.utils.set_random_seed(SEMILLA)
base_pre = construir_base("base_preentrenada")
modelo_aux = con_cabeza(base_pre, len(CLASES_FORMAS), dropout=0.0)
modelo_aux.fit(X_aux, y_aux, epochs=12, batch_size=64, verbose=0)
print("Exactitud en la tarea auxiliar (validación):", round(modelo_aux.evaluate(X_aux_v, y_aux_v, verbose=0)[1], 3))
print("La base tiene", base_pre.count_params(), "parámetros y ya sabe extraer rasgos visuales.")

### ✏️ A7 · EJERCICIO 1 de 2 — Congelar una red

**Tu turno:** Escribe `congelar(modelo)` que ponga `trainable = False` en **todas** las capas del modelo dado (es decir, `modelo.trainable = False`) y lo devuelva. Después crea `copia_base` (una copia independiente de `base_pre`, ya lo tienes hecho abajo) y congélala.

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Qué significa congelar una capa?
2. ¿Por qué se trabaja con una copia de la base?
3. ¿Qué ventaja tiene congelar la base?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
# pista: modelo.trainable = False y return modelo
def congelar(modelo):
    ...

def copiar_base(base, nombre):
    """Copia independiente de la base con los mismos pesos (para no modificar la original)."""
    copia = keras.models.clone_model(base)
    copia.set_weights(base.get_weights())
    copia._name = nombre
    return copia

copia_base = congelar(copiar_base(base_pre, "base_congelada"))
assert len(copia_base.trainable_weights) == 0, "Tras congelar no debe quedar ningún peso entrenable en la base"
assert len(copia_base.non_trainable_weights) == len(base_pre.weights), "Todos los pesos de la base deben pasar a 'no entrenables'"
assert len(base_pre.trainable_weights) > 0, "La base original no debe quedar congelada (usa una copia)"
print("✔ Ejercicio 1 correcto")

### 👀 A8 · EJEMPLO — Entrenar solo la cabeza (base congelada)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Cuántos parámetros se entrenan y cuántos tiene el modelo?
2. ¿Qué exactitud logra con la base congelada y cómo se compara con la CNN desde cero?
3. ¿Por qué no basta un solo entrenamiento para decidir?

In [ ]:
transfer = con_cabeza(copia_base, 5)
n_entrenables = sum(int(np.prod(p.shape)) for p in transfer.trainable_weights)
print(f"Parámetros totales: {transfer.count_params()} · entrenables (solo la cabeza): {n_entrenables}")
h_transfer = entrenar(transfer, epocas=30)
acc_transfer = transfer.evaluate(X_pru, y_pru, verbose=0)[1]
print(f"Transfer learning (base congelada) → exactitud en prueba: {acc_transfer:.3f}")

### ✏️ A9 · EJERCICIO 2 de 2 — Descongelar solo un bloque

**Tu turno:** Escribe `descongelar_bloque(base, n)` que ponga `trainable = True` **solo** en las capas de la base cuyo nombre empieza por `f"bloque{n}"` (por ejemplo `bloque3_conv`) y devuelva el número de pesos entrenables resultantes (`len(base.trainable_weights)`). Antes, `base.trainable` debe estar en `True` para poder cambiar capas sueltas, así que pon ese valor y después congela a mano las demás capas.

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Cuántos pesos entrenables tiene un bloque y por qué?
2. ¿Por qué el *fine-tuning* solo toca el último bloque?
3. ¿Cuál es el orden correcto de trabajo?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
# pista: base.trainable = True; recorre base.layers y pon trainable según el nombre de la capa
def descongelar_bloque(base, n):
    ...

_prueba = copiar_base(base_pre, "prueba")
assert descongelar_bloque(_prueba, 3) == 2, "El bloque 3 tiene una convolución con 2 pesos entrenables (filtros y sesgos)"
assert not _prueba.get_layer("bloque1_conv").trainable, "Los bloques 1 y 2 deben seguir congelados"
assert descongelar_bloque(_prueba, 2) == 2 and not _prueba.get_layer("bloque3_conv").trainable, "Solo debe estar activo el bloque indicado"
print("✔ Ejercicio 2 correcto")

### 👀 A10 · EJEMPLO — Fine-tuning del bloque 3 con tasa pequeña

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A10).

1. ¿Por qué hay que recompilar el modelo?
2. ¿Qué tasa se usa en el *fine-tuning* y por qué?
3. ¿Qué exactitud logra y qué conclusión sacas de un solo experimento?

In [ ]:
# Fine-tuning: partimos del modelo transfer ya entrenado (cabeza lista), descongelamos el bloque 3 y bajamos la tasa
n_ent_ft = descongelar_bloque(copia_base, 3)
transfer.compile(keras.optimizers.Adam(1e-4), "sparse_categorical_crossentropy", metrics=["accuracy"])   # ¡recompilar!
h_ft = entrenar(transfer, epocas=25, semilla=1)
acc_ft = transfer.evaluate(X_pru, y_pru, verbose=0)[1]
print(f"Tras fine-tuning del bloque 3 → exactitud en prueba: {acc_ft:.3f}")

### 👀 A11 · EJEMPLO — Comparar las tres estrategias con 3 semillas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A11).

1. ¿Qué media de exactitud tiene cada estrategia?
2. ¿Qué indica la desviación típica de cada una?
3. ¿Se puede afirmar que el *fine-tuning* es claramente mejor?

In [ ]:
def experimento(semilla, X=None, y=None, epocas=30):
    """Devuelve la exactitud en prueba de (desde cero, base congelada, base con fine-tuning)."""
    keras.utils.set_random_seed(semilla)
    m0 = con_cabeza(construir_base("cero"), 5)
    entrenar(m0, epocas + 10, semilla, X, y)
    b = congelar(copiar_base(base_pre, "b"))
    m1 = con_cabeza(b, 5)
    entrenar(m1, epocas, semilla, X, y)
    a1 = m1.evaluate(X_pru, y_pru, verbose=0)[1]
    descongelar_bloque(b, 3)
    m1.compile(keras.optimizers.Adam(1e-4), "sparse_categorical_crossentropy", metrics=["accuracy"])
    entrenar(m1, epocas - 5, semilla, X, y)
    return m0.evaluate(X_pru, y_pru, verbose=0)[1], a1, m1.evaluate(X_pru, y_pru, verbose=0)[1]

filas = [experimento(s) for s in (1, 2, 3)]
comparacion = pd.DataFrame(filas, columns=["Desde cero", "Base congelada", "Fine-tuning bloque 3"]).agg(["mean", "std"]).round(3)
comparacion

### 👀 A12 · EJEMPLO — Gráfico de barras de la comparación

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A12).

1. ¿Qué representan las barras de error?
2. ¿Por qué con ImageNet la mejora sería mayor?
3. ¿Qué se cuida con `plt.ylim(0, 1)`?

In [ ]:
comparacion.loc["mean"].plot(kind="bar", yerr=comparacion.loc["std"], capsize=4, color=["gray", "steelblue", "seagreen"])
plt.ylabel("exactitud en prueba"); plt.ylim(0, 1); plt.xticks(rotation=0)
plt.title(f"Con solo {N_POR_CLASE} imágenes por clase (media de 3 semillas)"); plt.show()

### 👀 A13 · EJEMPLO — Construir MobileNetV2 (sin pesos, sin internet)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A13).

1. ¿Cuántos parámetros se entrenan en este modelo y por qué tan pocos?
2. ¿Para qué sirve `training=False` al llamar a la base congelada?
3. ¿Qué cambia respecto a la rama offline y qué no?

In [ ]:
# Esta celda funciona SIN internet: construye MobileNetV2 con pesos aleatorios (weights=None) para ver su estructura.
def construir_mobilenet(pesos=None, tam=96):
    """pesos=None (aleatorios, offline) o pesos='imagenet' (requiere internet; descarga unos pocos MB, tamaño no verificado)."""
    base = keras.applications.MobileNetV2(input_shape=(tam, tam, 3), include_top=False, weights=pesos, alpha=0.35)
    base.trainable = False
    entrada = keras.layers.Input(shape=(48, 48, 3))
    x = keras.layers.Resizing(tam, tam)(entrada)                                     # de 48×48 a tam×tam
    x = keras.applications.mobilenet_v2.preprocess_input(x)                          # píxeles → [−1, 1]
    x = base(x, training=False)                                                      # ¡training=False al congelar!
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.2)(x)
    salida = keras.layers.Dense(5, activation="softmax")(x)
    modelo = keras.Model(entrada, salida)
    modelo.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo, base

mn, mn_base = construir_mobilenet(pesos=None)
print("MobileNetV2 (alpha 0.35): capas de la base =", len(mn_base.layers), "· parámetros totales =", mn.count_params(),
      "· entrenables =", sum(int(np.prod(p.shape)) for p in mn.trainable_weights))
print("Salida del modelo para 2 imágenes sintéticas:", mn.predict(X_ent[:2], verbose=0).round(2))

### 👀 A14 · EJEMPLO — Rama Colab (opcional, requiere internet)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A14).

1. ¿Qué hace la celda por defecto?
2. ¿Qué tasa se usa en el *fine-tuning* con pesos reales?

In [ ]:
USAR_INTERNET = False        # ← True solo con conexión (Colab)

if USAR_INTERNET:
    mn_real, mn_real_base = construir_mobilenet(pesos="imagenet")           # ← aquí se descargan los pesos
    mn_real.fit(X_ent, y_ent, validation_data=(X_val, y_val), epochs=15, batch_size=16, verbose=0)
    print("MobileNetV2 con ImageNet, exactitud en prueba:", round(mn_real.evaluate(X_pru, y_pru, verbose=0)[1], 3))
    # Fine-tuning: se descongelan las últimas capas de la base y se baja la tasa (recompilar siempre)
    mn_real_base.trainable = True
    for capa in mn_real_base.layers[:-20]:
        capa.trainable = False
    mn_real.compile(keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", metrics=["accuracy"])
    mn_real.fit(X_ent, y_ent, validation_data=(X_val, y_val), epochs=10, batch_size=16, verbose=0)
    print("Tras fine-tuning:", round(mn_real.evaluate(X_pru, y_pru, verbose=0)[1], 3))
else:
    print("Rama de Colab desactivada (USAR_INTERNET = False).")

### ✏️ A15 · MINI-RETO — ¿Cuántas fotos hay que etiquetar?

**Tu turno:** El cliente pregunta: *"¿Cuántas fotos por tipo de alojamiento tenemos que etiquetar?"* Responde con datos: entrena la **CNN desde cero** y el **transfer learning con base congelada** con `n` ∈ {3, 6, 12, 24} imágenes por clase (genéralas con `generar_lote(alojamiento, CLASES_ALOJ, n, semilla=20)`) y guarda en el diccionario `curva_datos` la exactitud en prueba de cada estrategia, con la forma `{n: (acc_desde_cero, acc_transfer)}`. Usa 40 épocas para "desde cero" y 30 para "transfer". Después, responde con tus palabras (🧠).

**Debe salir:** una gráfica de dos curvas (desde cero y transfer) con la exactitud en prueba según `n` y ✔ Mini-reto correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A15).

1. ¿Qué ventaja tiene el *transfer learning* con 3 imágenes por clase?
2. ¿Qué pasa con 24 imágenes por clase?
3. ¿Qué le recomendarías al cliente sobre cuántas fotos etiquetar?

In [ ]:
# pista: para cada n, genera datos con generar_lote(alojamiento, CLASES_ALOJ, n, semilla=20), entrena ambos modelos con entrenar(modelo, epocas, semilla, X, y) y evalúa en X_pru
curva_datos = {}
...

assert set(curva_datos) == {3, 6, 12, 24}, "Debe haber una entrada por cada n: 3, 6, 12 y 24"
assert all(len(v) == 2 and 0 <= v[0] <= 1 and 0 <= v[1] <= 1 for v in curva_datos.values()), "Cada valor debe ser (exactitud desde cero, exactitud transfer) entre 0 y 1"

ns = sorted(curva_datos)
plt.plot(ns, [curva_datos[n][0] for n in ns], "o-", label="desde cero")
plt.plot(ns, [curva_datos[n][1] for n in ns], "s-", label="transfer learning (base congelada)")
plt.xlabel("imágenes por clase"); plt.ylabel("exactitud en prueba"); plt.ylim(0, 1.02)
plt.title("¿Cuántas imágenes hacen falta?"); plt.legend(); plt.show()
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga `numpy`, `pandas`, `matplotlib`, `tensorflow`, `keras` y `functools` (para fijar argumentos de una función).
    - `os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"` silencia los mensajes técnicos de TensorFlow.
    - `keras.utils.set_random_seed(SEMILLA)` fija las semillas para que los resultados se repitan.
- **Qué debe salir:**
    - `Listo ✔`. Importar TensorFlow tarda unos segundos.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'np' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.
    - `ModuleNotFoundError: No module named 'tensorflow'` → fuera de Colab, instala TensorFlow en tu entorno.

## 0. Generador de imágenes sintéticas

#### A2 · Ejemplo — Cargar el generador de imágenes sintéticas

**Contexto TurisData.** Entrenar una CNN desde cero necesita miles de fotos etiquetadas, y el cliente tiene solo unos cientos. La solución habitual es **reutilizar una red ya entrenada** en otra tarea y adaptarla: es el *transfer learning*.

Estas funciones dibujan con PIL (i) **formas y texturas** (para el preentrenamiento) y (ii) **alojamientos
esquemáticos** (la tarea real). No hace falta que las estudies: ejecútala y sigue. Son dibujos, **no fotos reales**.

- **Qué hace:**
    - Define funciones que dibujan con PIL: `dibujar_forma` (8 formas y texturas, para el preentrenamiento) y `dibujar_alojamiento` (5 tipos de alojamiento esquemáticos: hotel, apartamentos, casa rural, camping y piscina, que es la tarea real).
    - `generar_lote(funcion, clases, n_por_clase, semilla)` devuelve las imágenes (`X`, de 48×48×3) y sus etiquetas (`y`), mezcladas.
    - También define `crear_dataset_carpetas`, que escribe imágenes en carpetas por clase (se usará en el reto del sprint).
- **Qué debe salir:**
    - No imprime nada: solo define las funciones.
- **Cómo leerlo:**
    - No hace falta estudiar el código de dibujo: son dibujos, **no fotos reales**. Lo importante son las dos tareas: formas y texturas (auxiliar) y tipos de alojamiento (real).
- **Si falla:**
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué dos tareas genera el notebook y para qué sirve cada una?* → **Formas y texturas** (8 clases) para **preentrenar** la base, y **alojamientos** esquemáticos (5 clases) que son la **tarea real**.
2. *¿Qué tamaño tienen las imágenes?* → **48×48 píxeles** con 3 canales de color.
3. *¿Son fotos reales?* → **No**: son dibujos sintéticos, para poder trabajar sin internet.

</details>

#### A3 · Ejemplo — Ver las dos tareas

- **Qué hace:**
    - `generar_lote(dibujar_forma, CLASES_FORMAS, 6, semilla=1)` y `generar_lote(dibujar_alojamiento, CLASES_ALOJ, 6, semilla=1)` crean 48 y 30 imágenes.
    - Se dibujan las 10 primeras de cada lote, con su clase como título.
- **Qué debe salir:**
    - Dos filas de 10 imágenes: arriba formas y texturas (rayas, cuadrículas, círculos…) y abajo alojamientos (hoteles, apartamentos, tiendas de campaña, piscinas…).
- **Cómo leerlo:**
    - Las dos tareas son **distintas pero relacionadas**: ambas usan bordes, esquinas y patrones repetidos.
- **Si falla:**
    - `NameError: name 'generar_lote' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué tipo de imágenes hay en la fila superior y en la inferior?* → Arriba, **formas y texturas** (tarea auxiliar); abajo, **alojamientos** (tarea real).
2. *¿Qué rasgos visuales comparten las dos tareas?* → **Bordes, esquinas y patrones repetidos**: por eso lo aprendido en una puede servir en la otra.
3. *¿Por qué se usa una tarea auxiliar sintética?* → Porque en el taller **no hay internet** ni pesos de ImageNet: preentrenamos la base con dibujos para ver la mecánica completa.

</details>

## 1. La tarea real, con pocos datos

#### A4 · Ejemplo — Crear el conjunto de entrenamiento, validación y prueba

**Idea.** Imagina que TurisData solo tiene **6 imágenes por tipo de alojamiento** (30 en total) etiquetadas y con mucho ruido.
Es muy poco para una CNN desde cero. Reservamos 200 para validación y 300 para prueba.

- **Qué hace:**
    - `N_POR_CLASE = 6` fija 6 imágenes por tipo de alojamiento (30 en total para entrenar).
    - `alojamiento` es `dibujar_alojamiento` con `ruido=40` (imágenes «sucias», como fotos de mala calidad), fijado con `functools.partial`.
    - `generar_lote(...)` crea el entrenamiento (6 por clase), la validación (40 por clase) y la prueba (60 por clase), cada uno con una semilla distinta.
- **Qué debe salir:**
    - `entrenamiento (30, 48, 48, 3) · validación (200, 48, 48, 3) · prueba (300, 48, 48, 3)`.
- **Cómo leerlo:**
    - 30 imágenes para entrenar son muy pocas para una CNN desde cero: es el caso donde el *transfer learning* puede ayudar.
- **Si falla:**
    - `NameError: name 'generar_lote' is not defined` → faltan los ejemplos del generador.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas imágenes hay para entrenar, validar y probar?* → **30**, **200** y **300**.
2. *¿Por qué las imágenes llevan tanto ruido?* → Para parecerse a **fotos de mala calidad** y hacer la tarea más difícil.
3. *¿Por qué cada conjunto usa una semilla distinta?* → Para que las imágenes de **validación y prueba sean distintas** de las de entrenamiento.

</details>

## 2. Punto de partida: una CNN desde cero

#### A5 · Ejemplo — Entrenar una CNN desde cero con 30 imágenes

La red se compone de una **base** (bloques `Conv → Pool` que extraen rasgos y terminan en
`Flatten`, que aplana los mapas en una lista de rasgos) y una **cabeza** (capa `Dense` que decide la clase).
Esta división es la clave del *transfer learning*: **la base se reutiliza; la cabeza se cambia**.

- **Qué hace:**
    - `construir_base(nombre)` crea la **base**: `Rescaling(1/255)` y tres bloques `Conv2D` + `MaxPooling2D` (16, 32 y 64 filtros) que terminan en `Flatten` (6×6×64 = 2 304 rasgos).
    - `con_cabeza(base, n_clases, dropout, tasa)` añade la **cabeza** (`Dropout` y `Dense` con softmax) y compila el modelo.
    - `entrenar(modelo, epocas, semilla, X, y)` fija la semilla y entrena con la validación; por defecto usa las 30 imágenes.
    - `desde_cero` es esa CNN con una base nueva (sin preentrenar), entrenada 40 épocas; `acc_cero` es su exactitud en prueba.
- **Qué debe salir:**
    - `CNN desde cero con 30 imágenes → exactitud en prueba: 0.800`.
- **Cómo leerlo:**
    - Acierta el 80 % en prueba con solo 30 imágenes de entrenamiento: es nuestro **punto de partida** para comparar. Con 5 clases, acertar al azar daría 0,2.
- **Si falla:**
    - `NameError: name 'X_ent' is not defined` → falta el ejemplo de los datos. Tarda unos segundos en entrenar.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué exactitud logra la CNN desde cero?* → **0,800** en prueba.
2. *¿Qué es la «base» y qué es la «cabeza»?* → La **base** son los bloques `Conv → Pool` que extraen rasgos (acaban en `Flatten`); la **cabeza** es la capa `Dense` que decide la clase.
3. *¿Por qué esta división es la clave del *transfer learning*?* → Porque **la base se reutiliza** y **la cabeza se cambia** según la nueva tarea.

</details>

## 3. Preentrenar la base en una tarea auxiliar

#### A6 · Ejemplo — Preentrenar la base con formas y texturas

En la vida real se descargaría una base entrenada con **millones de fotos** (ImageNet). Aquí, para poder trabajar sin
internet, la base aprende primero una tarea **distinta pero relacionada**: reconocer 8 clases de formas y
texturas (rayas, cuadrículas, círculos, triángulos…) con 3200 imágenes. Esos rasgos (bordes, esquinas, patrones
repetidos) también sirven para reconocer edificios y tiendas de campaña.

- **Qué hace:**
    - `generar_lote(dibujar_forma, CLASES_FORMAS, 400, ...)` crea 3 200 imágenes de formas y texturas para entrenar y 400 para validar.
    - `base_pre` es una base nueva; `modelo_aux` le pone una cabeza de 8 clases (las formas) y la entrena 12 épocas.
    - Los `print` muestran la exactitud en la tarea auxiliar y el número de parámetros de la base.
- **Qué debe salir:**
    - `Exactitud en la tarea auxiliar (validación): 0.938` y `La base tiene 23584 parámetros y ya sabe extraer rasgos visuales.`
- **Cómo leerlo:**
    - La base ha aprendido a reconocer las formas con un 93,8 % de exactitud: **ya sabe extraer bordes, esquinas y patrones**. Esa base es la que reutilizaremos.
- **Si falla:**
    - `NameError: name 'construir_base' is not defined` → falta el ejemplo anterior. Tarda unos segundos en entrenar.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos parámetros tiene la base?* → **23 584**.
2. *¿Qué exactitud logra en la tarea auxiliar?* → **0,938** (8 clases).
3. *¿En la vida real, con qué se preentrenaría la base?* → Con **millones de fotos** (por ejemplo ImageNet), no con 3 200 dibujos.

</details>

## 4. Congelar la base y entrenar solo una cabeza nueva

#### A7 · Ejercicio 1 — Congelar una red

**Idea.** "Congelar" una capa (`trainable = False`) significa que su peso **no se actualiza** al entrenar. Así
conservamos lo aprendido y solo se entrenan las pocas neuronas de la cabeza: hacen falta menos datos y menos tiempo.

- **Qué hace:**
    - `modelo.trainable = False` **congela** todas las capas del modelo: sus pesos ya no se actualizan al entrenar. La función devuelve el modelo.
    - `copiar_base(base, nombre)` (ya escrita) hace una copia independiente de la base con los mismos pesos, para no congelar la original.
    - `copia_base = congelar(copiar_base(base_pre, "base_congelada"))` congela la copia y las comprobaciones verifican que no queda ningún peso entrenable en ella y que la base original sigue entrenable.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `Tras congelar no debe quedar ningún peso entrenable en la base` → falta `modelo.trainable = False`.
    - `La base original no debe quedar congelada (usa una copia)` → has congelado `base_pre` en vez de una copia.
    - `TypeError: 'NoneType' object has no attribute ...` → falta `return modelo`.
    - `NameError: name 'base_pre' is not defined` → falta el ejemplo de preentrenar la base.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué significa congelar una capa?* → Que su peso **no se actualiza** al entrenar (`trainable = False`).
2. *¿Por qué se trabaja con una copia de la base?* → Para **no modificar la base original** (`base_pre`), que se usará otra vez.
3. *¿Qué ventaja tiene congelar la base?* → Se conserva lo aprendido y solo se entrenan las neuronas de la cabeza: hacen falta **menos datos y menos tiempo**.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def congelar(modelo):
    modelo.trainable = False
    return modelo
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A8 · Ejemplo — Entrenar solo la cabeza (base congelada)

- **Qué hace:**
    - `con_cabeza(copia_base, 5)` pone una cabeza nueva de 5 clases (los alojamientos) sobre la base congelada.
    - `n_entrenables` cuenta los parámetros que se pueden entrenar (solo los de la cabeza).
    - `entrenar(transfer, epocas=30)` entrena 30 épocas con las 30 imágenes y `acc_transfer` es su exactitud en prueba.
- **Qué debe salir:**
    - `Parámetros totales: 35109 · entrenables (solo la cabeza): 11525` y `Transfer learning (base congelada) → exactitud en prueba: 0.833`.
- **Cómo leerlo:**
    - Solo se entrenan **11 525 de 35 109** parámetros (la base de 23 584 está congelada). Con la base preentrenada se pasa del 0,800 de la CNN desde cero al **0,833**: mejora con menos parámetros que entrenar. Con un solo entrenamiento y 30 imágenes la diferencia es pequeña (se comprueba mejor con varias semillas, sección 5).
- **Si falla:**
    - `NameError: name 'copia_base' is not defined` → falta el ejercicio 1. Tarda unos segundos en entrenar.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos parámetros se entrenan y cuántos tiene el modelo?* → Se entrenan **11 525** de **35 109**.
2. *¿Qué exactitud logra con la base congelada y cómo se compara con la CNN desde cero?* → **0,833** frente a **0,800**.
3. *¿Por qué no basta un solo entrenamiento para decidir?* → Porque con **30 imágenes** el resultado es **muy variable**: hay que repetir con varias semillas.

</details>

## 5. Fine-tuning: descongelar las últimas capas

#### A9 · Ejercicio 2 — Descongelar solo un bloque

**Idea.** Una vez entrenada la cabeza, se puede **descongelar** el último bloque de la base y seguir entrenando con una
**tasa de aprendizaje muy pequeña** (10 veces menor o más): se ajustan los rasgos más específicos sin destrozar los generales.
Regla de oro: *primero cabeza, después fine-tuning, y con tasa pequeña*. Los primeros bloques (bordes, texturas) son
universales y casi nunca se tocan.

- **Qué hace:**
    - `base.trainable = True` activa la base para poder cambiar capas sueltas.
    - El bucle recorre `base.layers` y pone `capa.trainable = capa.name.startswith(f"bloque{n}")`: solo las capas del bloque `n` quedan entrenables.
    - La función devuelve `len(base.trainable_weights)`, el número de pesos entrenables (2 para un bloque: filtros y sesgos).
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`.
- **Cómo leerlo:**
    - Regla de oro del *fine-tuning*: **primero la cabeza, después descongelar el último bloque y con tasa pequeña**. Los primeros bloques (bordes, texturas) son universales y casi nunca se tocan.
- **Si falla:**
    - `El bloque 3 tiene una convolución con 2 pesos entrenables` → no pones `trainable = True` solo en las capas del bloque.
    - `Los bloques 1 y 2 deben seguir congelados` → has dejado entrenables todas las capas (falta poner `False` en las demás).
    - `Solo debe estar activo el bloque indicado` → al cambiar de bloque, el anterior sigue activo.
    - `NameError: name 'copiar_base' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos pesos entrenables tiene un bloque y por qué?* → **2**: los filtros y los sesgos de su capa convolucional.
2. *¿Por qué el *fine-tuning* solo toca el último bloque?* → Los primeros bloques detectan rasgos **universales** (bordes, texturas); el último es más específico de la tarea.
3. *¿Cuál es el orden correcto de trabajo?* → **Primero la cabeza, después el *fine-tuning* y con tasa pequeña.**

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def descongelar_bloque(base, n):
    base.trainable = True
    for capa in base.layers:
        capa.trainable = capa.name.startswith(f"bloque{n}")
    return len(base.trainable_weights)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A10 · Ejemplo — Fine-tuning del bloque 3 con tasa pequeña

- **Qué hace:**
    - `descongelar_bloque(copia_base, 3)` activa solo el último bloque de la base del modelo `transfer` (ya entrenado).
    - `transfer.compile(Adam(1e-4), ...)` **recompila** con una tasa diez veces menor (hay que recompilar siempre al cambiar qué capas se entrenan).
    - `entrenar(transfer, epocas=25, semilla=1)` sigue entrenando 25 épocas; `acc_ft` es la exactitud en prueba.
- **Qué debe salir:**
    - `Tras fine-tuning del bloque 3 → exactitud en prueba: 0.830`.
- **Cómo leerlo:**
    - Con esta semilla el fine-tuning queda en 0,830, casi igual que la base congelada (0,833): con tan pocos datos **un solo experimento no decide**. Hay que repetirlo, como en el siguiente ejemplo.
- **Si falla:**
    - `NameError: name 'descongelar_bloque' is not defined` → falta el ejercicio 2.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué hay que recompilar el modelo?* → Porque al cambiar qué capas son entrenables (y la tasa) el modelo debe **compilarse de nuevo** para que Keras lo tenga en cuenta.
2. *¿Qué tasa se usa en el *fine-tuning* y por qué?* → **1e-4**, diez veces menor: para ajustar los rasgos sin destrozar lo aprendido.
3. *¿Qué exactitud logra y qué conclusión sacas de un solo experimento?* → **0,830**, casi igual que antes: con pocos datos **un solo experimento no basta**.

</details>

#### A11 · Ejemplo — Comparar las tres estrategias con 3 semillas

Un solo entrenamiento con 30 imágenes es muy variable. Repetimos las tres estrategias con **3 semillas** (la base
preentrenada es la misma; cambia la inicialización de la cabeza y el orden de los lotes).

- **Qué hace:**
    - `experimento(semilla)` entrena las tres estrategias —desde cero (40 épocas), base congelada (30) y *fine-tuning* del bloque 3 (25 más)— y devuelve la exactitud en prueba de cada una.
    - `filas` repite el experimento con las semillas 1, 2 y 3; `comparacion` es una tabla con la media y la desviación típica de cada estrategia.
- **Qué debe salir:**
    - Una tabla con `mean` y `std`: «Desde cero» 0,808 (± 0,028) · «Base congelada» 0,821 (± 0,014) · «Fine-tuning bloque 3» 0,839 (± 0,008). Tarda un par de minutos.
- **Cómo leerlo:**
    - La media sube de 0,808 a 0,821 y a 0,839, y la variación entre semillas **baja** (0,028 → 0,014 → 0,008): la base preentrenada da un resultado mejor y más estable. Las diferencias son pequeñas (1–3 puntos) con solo 3 semillas: es una **tendencia**, no una prueba definitiva.
- **Si falla:**
    - `NameError: name 'descongelar_bloque' is not defined` → falta el ejercicio 2. Tarda varios minutos: es normal.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué media de exactitud tiene cada estrategia?* → Desde cero **0,808**, base congelada **0,821** y *fine-tuning* **0,839**.
2. *¿Qué indica la desviación típica de cada una?* → Lo **estable** que es el resultado entre semillas: baja de 0,028 (desde cero) a 0,008 (*fine-tuning*).
3. *¿Se puede afirmar que el *fine-tuning* es claramente mejor?* → **No del todo**: la diferencia es de 1–3 puntos con 3 semillas; es una tendencia, no una prueba definitiva.

</details>

#### A12 · Ejemplo — Gráfico de barras de la comparación

**Cómo leerlo.** Con pocas imágenes, la base preentrenada suele dar mejor punto de partida que aprender todo desde cero,
y el *fine-tuning* añade un pequeño ajuste (o no: con muy pocos datos puede sobreajustar). Los resultados exactos varían;
lo importante es el método. **Con pesos reales de ImageNet la diferencia es mucho mayor**, porque esa base ha visto
millones de fotos de objetos reales, no 3200 dibujos.

- **Qué hace:**
    - `comparacion.loc["mean"].plot(kind="bar", yerr=comparacion.loc["std"], ...)` dibuja la media de cada estrategia con una barra de error de una desviación típica.
- **Qué debe salir:**
    - Un gráfico de tres barras (gris, azul y verde) entre 0,8 y 0,84 con pequeñas barras de error.
- **Cómo leerlo:**
    - Las barras tienen alturas parecidas: la mejora es modesta. **Con pesos reales de ImageNet la diferencia es mucho mayor**, porque esa base ha visto millones de fotos de objetos reales, no 3 200 dibujos.
- **Si falla:**
    - `NameError: name 'comparacion' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué representan las barras de error?* → Una **desviación típica** de la exactitud entre las 3 semillas.
2. *¿Por qué con ImageNet la mejora sería mayor?* → Porque esa base ha visto **millones de fotos reales**, no 3 200 dibujos.
3. *¿Qué se cuida con `plt.ylim(0, 1)`?* → Que el gráfico **no exagere** diferencias pequeñas: el eje parte de 0 hasta 1.

</details>

## 6. Con pesos reales: MobileNetV2

#### A13 · Ejemplo — Construir MobileNetV2 (sin pesos, sin internet)

**Qué cambia con pesos reales de ImageNet:**

| Aspecto | Rama offline (este notebook) | Con MobileNetV2 + ImageNet |
|---|---|---|
| Base | 3 bloques, ≈ 23 600 parámetros, preentrenada en 3200 dibujos | ≈ 2,2 millones de parámetros con `alpha` 1,0 (≈ 0,4 millones con la `alpha=0.35` de la celda de abajo), entrenada con 1,2 M de fotos |
| Tamaño de entrada | 48×48 | 96, 128, 160 o 224 (mínimo 32; con 224 es la configuración estándar) |
| Preprocesado | `Rescaling(1/255)` dentro del modelo | `keras.applications.mobilenet_v2.preprocess_input` (píxeles a [−1, 1]) |
| Normalización | sin `BatchNormalization` | la base lleva *batch norm*: al congelarla se llama con `training=False` |
| Resultado | mejora modesta | suele bastar con 20–100 fotos por clase para resultados útiles (orientativo; no verificado en el taller) |

La **estructura no cambia**: base congelada → cabeza nueva → fine-tuning con tasa pequeña.

- **Qué hace:**
    - `construir_mobilenet(pesos, tam)` carga `MobileNetV2` (con `alpha=0.35`, una versión pequeña) como base sin su cabeza original (`include_top=False`) y la congela.
    - El modelo redimensiona la imagen de 48×48 a `tam×tam` (`Resizing`), la preprocesa a [−1, 1], pasa por la base con `training=False`, aplica `GlobalAveragePooling2D`, `Dropout` y una cabeza de 5 clases.
    - Aquí se usa `pesos=None` (pesos aleatorios, sin internet) solo para ver la **estructura**.
- **Qué debe salir:**
    - `MobileNetV2 (alpha 0.35): capas de la base = 154 · parámetros totales = 416613 · entrenables = 6405` y una salida de `[0.2 0.2 0.2 0.2 0.2]` para cada una de las 2 imágenes.
- **Cómo leerlo:**
    - La base tiene 154 capas y ≈ 0,4 millones de parámetros, pero solo se entrenan **6 405** (la cabeza). Con pesos aleatorios la red no sabe nada: reparte la probabilidad por igual entre las 5 clases (0,2 cada una).
- **Si falla:**
    - `NameError: name 'X_ent' is not defined` → faltan los ejemplos anteriores.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos parámetros se entrenan en este modelo y por qué tan pocos?* → **6 405**: solo la cabeza; la base está congelada.
2. *¿Para qué sirve `training=False` al llamar a la base congelada?* → Para que sus capas de *batch normalization* **no actualicen sus estadísticas** al entrenar la cabeza.
3. *¿Qué cambia respecto a la rama offline y qué no?* → Cambian el **preprocesado**, el **tamaño de entrada** y `training=False`; **no cambia la estructura**: base congelada → cabeza nueva → *fine-tuning*.

</details>

#### A14 · Ejemplo — Rama Colab (opcional, requiere internet)

> **AVISO: la siguiente celda requiere internet y NO se ha verificado en el taller.** Descarga los pesos de ImageNet de
> MobileNetV2 (unos pocos MB para `alpha=0.35`; tamaño no verificado) y entrena la cabeza sobre las imágenes sintéticas de este notebook (solo para ver
> la mecánica; el reto usará tus fotos). Pon `USAR_INTERNET = True` en Colab.

- **Qué hace:**
    - Con `USAR_INTERNET = False` (por defecto) la celda solo escribe un mensaje. Con `True` descarga los pesos de ImageNet, entrena la cabeza 15 épocas y luego hace *fine-tuning* de las últimas 20 capas con tasa 1e-5.
- **Qué debe salir:**
    - `Rama de Colab desactivada (USAR_INTERNET = False).`
- **Cómo leerlo:**
    - Esta rama **no se ha verificado** sin internet. Para probarla, en Colab, cambia `USAR_INTERNET` a `True`.
- **Si falla:**
    - Con `USAR_INTERNET = True` sin conexión → error de descarga: déjalo en `False`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué hace la celda por defecto?* → Solo escribe el mensaje de que la rama está desactivada.
2. *¿Qué tasa se usa en el *fine-tuning* con pesos reales?* → **1e-5**, aún menor que en la rama offline.

</details>

## 7. Mini-reto: ¿cuántas imágenes necesito?

#### A15 · Mini-reto — ¿Cuántas fotos hay que etiquetar?

- **Qué hace:**
    - Para cada `n` (3, 6, 12 y 24 imágenes por clase), `generar_lote` crea los datos, se entrena una CNN **desde cero** (40 épocas) y un modelo con **base congelada** (30 épocas) y se evalúan con `X_pru`.
    - `curva_datos[n] = (acc_desde_cero, acc_transfer)` guarda las dos exactitudes de cada `n`. La gráfica las compara.
- **Qué debe salir:**
    - Una gráfica con dos curvas y `✔ Mini-reto correcto`. Con la solución de referencia: n = 3 → (0,737 · 0,817); n = 6 → (0,843 · 0,873); n = 12 → (0,917 · 0,957); n = 24 → (0,977 · 0,977).
- **Cómo leerlo:**
    - El *transfer learning* gana con pocos datos (3, 6 y 12 por clase) y la ventaja **desaparece** con 24 imágenes por clase, donde ambos llegan a 0,977. Tus valores pueden variar algo.
- **Si falla:**
    - `Debe haber una entrada por cada n: 3, 6, 12 y 24` → faltan valores de `n` en `curva_datos`.
    - `Cada valor debe ser (exactitud desde cero, exactitud transfer) entre 0 y 1` → cada valor debe ser una pareja de exactitudes.
    - `NameError: name 'generar_lote' is not defined` → faltan los ejemplos anteriores. Tarda un par de minutos.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué ventaja tiene el *transfer learning* con 3 imágenes por clase?* → **0,817** frente a **0,737** desde cero: ≈ 8 puntos más.
2. *¿Qué pasa con 24 imágenes por clase?* → La ventaja **desaparece**: ambos logran **0,977**.
3. *¿Qué le recomendarías al cliente sobre cuántas fotos etiquetar?* → Que con transfer learning **bastan pocas** (del orden de 12 por clase ya da ≈ 0,96 en este caso sintético) y que más de 24 apenas añade; con datos reales conviene verificarlo con una curva así.

</details>

## 8. Preguntas de reflexión
1. ¿Por qué se cambia la cabeza pero se reutiliza la base?
2. ¿Por qué el *fine-tuning* se hace con una tasa de aprendizaje mucho más pequeña y **después** de entrenar la cabeza?
3. Si tus fotos son muy distintas de las de ImageNet (por ejemplo, imágenes térmicas o de satélite), ¿el *transfer learning* seguirá ayudando? ¿Y descongelar más capas?
4. ¿Qué riesgo hay en usar un modelo preentrenado que no has entrenado tú (sesgos, licencias, datos con los que se entrenó)?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Porque la **base** ha aprendido rasgos generales (bordes, texturas, formas) que valen para casi cualquier imagen, mientras que la **cabeza** es la parte específica de la tarea original (sus clases). Reutilizar la base ahorra datos y tiempo; la cabeza hay que cambiarla porque las clases son otras.
2. Porque la cabeza nueva empieza con **pesos aleatorios**: si se descongelara la base desde el principio, los errores grandes de la cabeza «destrozarían» los rasgos ya aprendidos. Por eso se entrena **primero la cabeza**, y después se ajusta la base con una tasa muy pequeña para no perder lo aprendido.
3. Probablemente **ayudará menos**, porque los rasgos de ImageNet (objetos cotidianos) se parecen poco a imágenes térmicas o de satélite. Conviene **descongelar más capas** (y usar más datos propios) para adaptar rasgos más generales.
4. Los **sesgos** del conjunto con el que se entrenó, la **licencia** de uso del modelo y de sus datos, y la falta de **control** sobre qué ha aprendido (o sobre datos personales o protegidos presentes en su entrenamiento).

</details>

## 9. Resumen: qué has aprendido
- *Transfer learning* = reutilizar la **base** de una red preentrenada y entrenar una **cabeza** nueva.
- Se congela (`trainable = False`), se entrena la cabeza y, si hace falta, se hace *fine-tuning* de las últimas capas con tasa baja (y recompilando).
- Ayuda sobre todo con **pocos datos**; su ventaja depende de cuánto se parezcan las tareas.
- Con MobileNetV2/ImageNet cambian el preprocesado, el tamaño de entrada y el uso de `training=False`, pero no la estructura.